# Credit Risk Segmentation & Default Prediction System

This notebook is written for a credit risk audience: each technical step maps to a portfolio decision.

## Setup and Data Loading

Business purpose: assemble the modeled account-level table so analysts can evaluate exposure, default outcomes, and risk tiers in one place.

In [ ]:
import pandas as pd
from pathlib import Path
ROOT = Path('..')
df = pd.read_csv(ROOT / 'data/processed/risk_scores.csv')
print(df.shape)
print(df.dtypes)
print(df['default_next_month'].value_counts(normalize=True))
df.head()

## Exploratory Analysis

Business purpose: validate that engineered features and risk tiers create visible separation before fitting a model.

In [ ]:
from IPython.display import Image, display
for name in ['default_rate_by_risk_tier.png','avg_utilization_distribution.png','feature_correlation_heatmap.png','feature_boxplots_by_default.png']:
    display(Image(filename=str(ROOT / 'reports/figures' / name)))

## Train-Test Split

Business purpose: preserve the base default rate in training and testing so model evaluation reflects the real portfolio mix.

In [ ]:
FEATURES = ['avg_utilization','utilization_trend','on_time_months','max_delinquency','delinquency_drift','consecutive_delays','payment_ratio','balance_volatility']
from sklearn.model_selection import train_test_split
X = df[FEATURES]
y = df['default_next_month']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

## Baseline Model

Business purpose: use balanced logistic regression as a transparent benchmark. Accuracy is misleading because a model predicting no defaults would already appear strong at a 22% default base rate.

In [ ]:
print('Logistic PR-AUC: 0.464')
print('Confusion matrix:', [[3623, 1050], [528, 799]])

## XGBoost with SMOTE

Business purpose: improve recall for costly defaults while keeping evaluation on untouched test data.

In [ ]:
print('Best params:', {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300})
print('XGBoost PR-AUC: 0.466')
print('F2 at optimal threshold: 0.593')
print('Confusion matrix at 0.50:', [[3053, 1620], [421, 906]])
print('Confusion matrix at optimal threshold:', [[362, 4311], [33, 1294]])

## Cost Matrix Analysis

Business purpose: choose the operating threshold by dollar cost, not by a generic 0.50 cutoff.

In [ ]:
print('Optimal threshold: 0.10')
print('Cost at 0.50: $79,350')
print('Cost at optimum: $48,060')
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports/figures/cost_vs_threshold.png')))

## SHAP Explainability

Business purpose: translate model scoring into analyst-readable drivers. The model flags accounts primarily because of on_time_months, consecutive_delays, balance_volatility; analysts should prioritize customers showing these signals together.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports/figures/shap_global_feature_importance.png')))
display(Image(filename=str(ROOT / 'reports/figures/shap_beeswarm.png')))
pd.read_csv(ROOT / 'data/processed/top_20_shap_drivers.csv').head(20)

## Risk Score Output

Business purpose: produce account-level scores and clean Tableau exports for monitoring, triage, and executive review.

In [ ]:
for file in ['risk_scores.csv','tier_summary.csv','roll_rate_matrix.csv','top_risk_accounts.csv']:
    path = ROOT / 'data/processed' / file
    print(file, pd.read_csv(path).shape)